In [1]:
import pandas as pd
import json
import pickle
import gzip

version = "train_21"

In [ ]:
import yaml
from pathlib import Path

DATASET_CONFIG_PATH = Path("../../src/configs/datasets/transactions_21.yaml")
with open(DATASET_CONFIG_PATH) as f:
    DATASET_CONFIG = yaml.safe_load(f)

def validate_feature_sizes(df, feature_sizes, expected_features, dict_label):
    missing_in_dict = [feat for feat in expected_features if feat not in feature_sizes]
    extra_in_dict = [feat for feat in feature_sizes if feat not in expected_features]
    if missing_in_dict or extra_in_dict:
        raise ValueError(f"{dict_label} mismatch with dataset config. Missing: {missing_in_dict}, extra: {extra_in_dict}")

    missing_columns = [feat for feat in expected_features if feat not in df.columns]
    if missing_columns:
        raise ValueError(f"Columns not found in dataframe: {missing_columns}")

    violations = []
    for feat in expected_features:
        series = df[feat]
        max_val = series.max(skipna=True)
        if pd.isna(max_val):
            continue
        max_val = int(max_val)
        size = int(feature_sizes[feat])
        if max_val >= size:
            violations.append((feat, max_val, size))

    if violations:
        details = "; ".join([f"{name}: max {mx} >= size {sz}" for name, mx, sz in violations])
        raise ValueError(f"{dict_label} is too small for: {details}")

    print(f"Validated {dict_label}: config features match and sizes cover observed max indices.")


In [2]:
# dt features
import pandas as pd
import json
import pickle
import gzip

dt_features = pd.read_parquet(f"/home/datalab/nfs/deepfm/data/{version}/arnsdpsbx_t_team_fin_adviser.dt_pass_indexed")
cat_cols = [f for f in dt_features.columns if f.endswith("_index")]
feature_cols = list(dt_features.columns)

dt_features_dict = {
    row["inn_dt_index"]: {feat: row[feat] for feat in feature_cols}
    for _, row in dt_features.iterrows()
}

with gzip.open(f"/home/datalab/nfs/deepfm/data/{version}/dt_features_dict.pkl.gz", "wb") as fout:
    pickle.dump(dt_features_dict, fout)

user_feature_sizes = {col: dt_features[col].nunique() for col in cat_cols}
with open(f"/home/datalab/nfs/deepfm/data/{version}/user_feature_sizes.json", "w") as fout:
    json.dump(user_feature_sizes, fout, indent=4)
validate_feature_sizes(dt_features, user_feature_sizes, DATASET_CONFIG['train']['dt_features'], 'user_feature_sizes')


In [3]:
dt_features.head()

,inn_dt,dt_avg_sum,dt_stddev_sum,dt_min_sum,dt_max_sum,dt_median_sum,dt_buyers_count,dt_skewness_sum,inn_dt_index,okved_cd_dt_index,...,bic_dt_34_index,bic_dt_56_index,bic_dt_79_index,num_dt_13_index,num_dt_45_index,num_dt_68_index,okved_cd_dt_lvl1_index,okved_cd_dt_lvl2_index,okved_cd_dt_lvl3_index,okved_cd_dt_lvl4_index
0,censored,49270.1176562500000000000000,40719.877804,800.000000000000000000,130055.000000000000000000,38251.000000000000000000,18,0.434358,35.0,39.0,...,6.0,8.0,0.0,2.0,0.0,0.0,35.0,2.0,0.0,0.0
1,censored,11250.0000000000000000000000,7705.517504,5625.000000000000000000,20000.000000000000000000,5625.000000000000000000,1,0.410765,112.0,18.0,...,6.0,8.0,22.0,0.0,0.0,0.0,2.0,4.0,0.0,0.0
2,censored,45345.1045454545454545450000,24618.435841,1975.650000000000000000,73980.000000000000000000,44370.000000000000000000,5,-0.377609,272.0,271.0,...,6.0,8.0,0.0,1.0,0.0,0.0,8.0,26.0,1.0,0.0
3,censored,26860.8333333333333333330000,19753.123200,3600.000000000000000000,59053.000000000000000000,22804.000000000000000000,2,0.465418,312.0,149.0,...,6.0,8.0,53.0,0.0,0.0,0.0,8.0,19.0,0.0,0.0
4,censored,6091.5673076923076923080000,3622.921751,1600.000000000000000000,16926.000000000000000000,5001.000000000000000000,6,1.269281,440.0,43.0,...,4.0,7.0,0.0,2.0,0.0,0.0,0.0,14.0,0.0,0.0


In [3]:
# kt features
import pandas as pd
import json
import pickle
import gzip

kt_features = pd.read_parquet("/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.kt_pass_indexed")
cat_cols = [f for f in kt_features.columns if f.endswith("_index")]
feature_cols = list(kt_features.columns)

kt_features_dict = {
    row["inn_kt_index"]: {feat: row[feat] for feat in feature_cols}
    for _, row in kt_features.iterrows()
}

with gzip.open("/home/datalab/nfs/deepfm/data/train_21/kt_features_dict.pkl.gz", "wb") as fout:
    pickle.dump(kt_features_dict, fout)

item_feature_sizes = {col: kt_features[col].nunique() for col in cat_cols}
with open("/home/datalab/nfs/deepfm/data/train_21/item_feature_sizes.json", "w") as fout:
    json.dump(item_feature_sizes, fout, indent=4)
validate_feature_sizes(kt_features, item_feature_sizes, DATASET_CONFIG['train']['kt_features'], 'item_feature_sizes')


In [3]:
kt_features.head()

,inn_kt,kt_avg_sum,kt_stddev_sum,kt_min_sum,kt_max_sum,kt_median_sum,kt_buyers_count,kt_skewness_sum,inn_kt_index,okved_cd_kt_index,...,bic_kt_34_index,bic_kt_56_index,bic_kt_79_index,num_kt_13_index,num_kt_45_index,num_kt_68_index,okved_cd_kt_lvl1_index,okved_cd_kt_lvl2_index,okved_cd_kt_lvl3_index,okved_cd_kt_lvl4_index
0,censored,887293.3600000000000000000000,577883.407272,500000.000000000000000000,1891500.000000000000000000,750000.000000000000000000,2,1.293543,44.0,171.0,...,4.0,7.0,0.0,0.0,0.0,0.0,31.0,3.0,0.0,0.0
1,censored,222082.1428571428571428570000,204464.967478,12500.000000000000000000,588450.000000000000000000,118000.000000000000000000,10,0.749555,136.0,1.0,...,4.0,7.0,15.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0
2,censored,5300.0000000000000000000000,NaN,5300.000000000000000000,5300.000000000000000000,5300.000000000000000000,1,NaN,177.0,96.0,...,6.0,9.0,0.0,1.0,4.0,0.0,1.0,19.0,0.0,0.0
3,censored,36002.9761904761904761900000,24126.189778,250.000000000000000000,136000.000000000000000000,30000.000000000000000000,41,1.686667,444.0,0.0,...,4.0,7.0,0.0,1.0,1.0,0.0,2.0,1.0,0.0,0.0
4,censored,282800.0000000000000000000000,NaN,282800.000000000000000000,282800.000000000000000000,282800.000000000000000000,1,NaN,649.0,72.0,...,6.0,9.0,0.0,1.0,1.0,0.0,27.0,25.0,0.0,0.0


In [4]:
# test dict
import pandas as pd
import json
import pickle
import gzip

test = pd.read_parquet("/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.test_dict")

test_dict = {
    row["inn_dt_index"]: row["kt_set"]
    for _, row in test.iterrows()
}

with gzip.open("/home/datalab/nfs/deepfm/data/train_21/test_dict.pkl.gz", "wb") as fout:
    pickle.dump(test_dict, fout)

In [ ]:
# reverse test dict
import pandas as pd
import pickle
import gzip

reverse = pd.read_parquet(f"/home/datalab/nfs/deepfm/data/{version}/arnsdpsbx_t_team_fin_adviser.test_reverse_dict")

test_reverse_dict = {
    row["inn_kt_index"]: [int(dt) for dt in row["dt_set"]]
    for _, row in reverse.iterrows()
}

with gzip.open(f"/home/datalab/nfs/deepfm/data/{version}/test_reverse_dict.pkl.gz", "wb") as fout:
    pickle.dump(test_reverse_dict, fout)



In [5]:
# kt embeddings

import pandas as pd
import json
import pickle
import gzip

kt_embeddings = pd.read_parquet("/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.kt_embeddings_indexed")
kt_embeddings_dict = {
    row["inn_kt_index"]: row["embedding"]
    for _, row in kt_embeddings.iterrows()
}

with gzip.open("/home/datalab/nfs/deepfm/data/train_21/kt_embeddings_dict.pkl.gz", "wb") as fout:
    pickle.dump(kt_embeddings_dict, fout, protocol=pickle.HIGHEST_PROTOCOL)

In [6]:
# dt embeddings

dt_embeddings = pd.read_parquet("/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.dt_embeddings_indexed")
dt_embeddings_dict = {
    row["inn_dt_index"]: row["embedding"]
    for _, row in dt_embeddings.iterrows()
}

with gzip.open("/home/datalab/nfs/deepfm/data/train_21/dt_embeddings_dict.pkl.gz", "wb") as fout:
    pickle.dump(dt_embeddings_dict, fout, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import pandas as pd
import json, pickle, gzip
import os, sys

In [7]:
#mapping from inn_kt to inn_kt_index
import pyarrow.parquet as pq
import pandas as pd
import gzip
import pickle

dt_indexer_path = "/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.indexer_dt/data"
labels_df = pq.read_table(dt_indexer_path).to_pandas()
labels_array = labels_df["labelsArray"][0]
inn_dt_to_index = {
    labels_array[0][i]: i for i in range(len(labels_array[0]))
}

with gzip.open("/home/datalab/nfs/deepfm/data/train_21/inn_dt_to_index.pkl.gz", "wb") as fout:
    pickle.dump(inn_dt_to_index, fout)

In [8]:
# mapping from inn_kt to inn_kt_index
import pyarrow.parquet as pq
import pandas as pd
import gzip
import pickle

kt_indexer_path = "/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.indexer_kt/data"
labels_df = pq.read_table(kt_indexer_path).to_pandas()
labels_array = labels_df["labelsArray"][0]
inn_kt_to_index = {
    labels_array[0][i]: i for i in range(len(labels_array[0]))
}

with gzip.open("/home/datalab/nfs/deepfm/data/train_21/inn_kt_to_index.pkl.gz", "wb") as fout:
    pickle.dump(inn_kt_to_index, fout)

In [9]:
import pyarrow.parquet as pq
import pandas as pd
import gzip
import pickle

kt_indexer_path = "/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.indexer_kt/data"
labels_df = pq.read_table(kt_indexer_path).to_pandas()
labels_array = labels_df["labelsArray"][0]
inn_kt_to_index = {
    labels_array[0][i]: i for i in range(len(labels_array[0]))
}

dt_indexer_path = "/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.indexer_dt/data"
labels_df = pq.read_table(dt_indexer_path).to_pandas()
labels_array = labels_df["labelsArray"][0]
inn_dt_to_index = {
    labels_array[0][i]: i for i in range(len(labels_array[0]))
}

# unique dt list
dt_unique = list(inn_dt_to_index.keys())
with gzip.open("/home/datalab/nfs/deepfm/data/train_21/unique_inn_dt.pkl.gz", "wb") as f:
    pickle.dump(dt_unique, f, protocol=pickle.HIGHEST_PROTOCOL)
    
# unique kt list
kt_unique = list(inn_kt_to_index.keys())
with gzip.open("/home/datalab/nfs/deepfm/data/train_21/unique_inn_kt.pkl.gz", "wb") as f:
    pickle.dump(kt_unique, f, )

In [ ]:
# topic embeddings for dt
import pandas as pd
import numpy as np
import gzip
import pickle

embedding_path = f"/home/datalab/nfs/deepfm/data/{version}/arnsdpsbx_t_team_fin_adviser.dt_topic_embeddings"
dt_topics = pd.read_parquet(embedding_path)
topic_cols = [c for c in dt_topics.columns if c.startswith("dt_topic_weight_")]

dt_topic_embeddings = {}
for _, row in dt_topics.iterrows():
    inn_dt = row["inn_dt"]
    idx = inn_dt_to_index.get(inn_dt)
    if idx is None:
        continue
    dt_topic_embeddings[idx] = np.array([row[c] for c in topic_cols], dtype=np.float32)

with gzip.open(f"/home/datalab/nfs/deepfm/data/{version}/dt_topic_embeddings_dict.pkl.gz", "wb") as fout:
    pickle.dump(dt_topic_embeddings, fout, protocol=pickle.HIGHEST_PROTOCOL)


In [ ]:
# topic embeddings for kt
import pandas as pd
import numpy as np
import gzip
import pickle

embedding_path = f"/home/datalab/nfs/deepfm/data/{version}/arnsdpsbx_t_team_fin_adviser.kt_topic_embeddings"
kt_topics = pd.read_parquet(embedding_path)
topic_cols = [c for c in kt_topics.columns if c.startswith("kt_topic_weight_")]

kt_topic_embeddings = {}
for _, row in kt_topics.iterrows():
    inn_kt = row["inn_kt"]
    idx = inn_kt_to_index.get(inn_kt)
    if idx is None:
        continue
    kt_topic_embeddings[idx] = np.array([row[c] for c in topic_cols], dtype=np.float32)

with gzip.open(f"/home/datalab/nfs/deepfm/data/{version}/kt_topic_embeddings_dict.pkl.gz", "wb") as fout:
    pickle.dump(kt_topic_embeddings, fout, protocol=pickle.HIGHEST_PROTOCOL)
